# Divided Attention — Task Notebook

By Conrad Kleykamp

Task-compatible version of the Divided Attention benchmark. Runs against a single model provided by the Kaggle task runner (`kbench.llm`). To evaluate additional models, use the **Run on new model** option on the task page.

Evaluates whether models can simultaneously monitor and reason across two independent information streams, keeping each stream's facts correctly attributed without conflation. A fixed judge model (`kbench.judge_llm`) scores all responses to ensure consistent cross-model evaluation.

Each task row contains 5 judge-evaluated criteria. Total assertions: 25.

In [1]:
import kaggle_benchmarks as kbench
import pandas as pd

In [2]:
TASK_DATA = pd.DataFrame([
    {
        "stream_a": (
            "Server A logs: "
            "09:00 - Request received from user_42. "
            "09:02 - Query executed successfully. "
            "09:05 - Request received from user_17. "
            "09:07 - ERROR: Timeout on database connection. "
            "09:10 - Request received from user_99. "
            "09:12 - Query executed successfully. "
            "09:15 - ERROR: Memory limit exceeded. "
            "09:18 - Request received from user_42. "
            "09:20 - Query executed successfully."
        ),
        "stream_b": (
            "Server B logs: "
            "09:01 - Request received from user_55. "
            "09:03 - Query executed successfully. "
            "09:06 - Request received from user_23. "
            "09:08 - Query executed successfully. "
            "09:11 - ERROR: Disk read failure. "
            "09:13 - Request received from user_77. "
            "09:16 - Query executed successfully. "
            "09:19 - ERROR: Timeout on database connection. "
            "09:21 - Request received from user_55."
        ),
        "question": (
            "Monitoring both server logs simultaneously: "
            "How many errors occurred on each server, "
            "and which error type appeared on both servers?"
        ),
        "criteria": [
            "Response correctly identifies 2 errors on Server A.",
            "Response correctly identifies 2 errors on Server B.",
            "Response identifies 'Timeout on database connection' as appearing on both servers.",
            "Response does not confuse errors between the two servers.",
            "Response addresses both streams without omitting either.",
        ],
    },
    {
        "stream_a": (
            "Portfolio A trades: "
            "Trade 1: Bought 50 shares of AAPL at $178. "
            "Trade 2: Sold 30 shares of GOOG at $142. "
            "Trade 3: Bought 100 shares of MSFT at $415. "
            "Trade 4: Sold 50 shares of AAPL at $185. "
            "Trade 5: Bought 20 shares of AMZN at $192."
        ),
        "stream_b": (
            "Portfolio B trades: "
            "Trade 1: Bought 80 shares of TSLA at $245. "
            "Trade 2: Bought 40 shares of GOOG at $140. "
            "Trade 3: Bought 40 shares of NVDA at $620. "
            "Trade 4: Sold 80 shares of TSLA at $261. "
            "Trade 5: Sold 40 shares of NVDA at $875. "
            "Trade 6: Bought 60 shares of MSFT at $418."
        ),
        "question": (
            "Tracking both portfolios simultaneously: "
            "Which stock appears in both portfolios, "
            "and which portfolio realized a profit on its sold positions?"
        ),
        "criteria": [
            "Response correctly identifies GOOG and MSFT as appearing in both portfolios.",
            "Response correctly identifies Portfolio A realized profit on AAPL (bought at $178, sold at $185).",
            "Response correctly identifies Portfolio B realized profit on TSLA (bought at $245, sold at $261).",
            "Response correctly identifies Portfolio B realized profit on NVDA sold position.",
            "Response does not conflate trades between the two portfolios.",
        ],
    },
    {
        "stream_a": (
            "Team A sprint updates: "
            "Monday: Completed user authentication module. "
            "Tuesday: Started API integration, blocked on missing credentials. "
            "Wednesday: Credentials received, API integration resumed. "
            "Thursday: API integration completed and tested. "
            "Friday: Began dashboard UI development."
        ),
        "stream_b": (
            "Team B sprint updates: "
            "Monday: Completed database schema design. "
            "Tuesday: Started data migration scripts. "
            "Wednesday: Data migration 60% complete, no blockers. "
            "Thursday: Data migration completed. "
            "Friday: Started integration testing with Team A's API."
        ),
        "question": (
            "Tracking both team streams simultaneously: "
            "Which team experienced a blocker this sprint, "
            "and on which day did the two teams' work become directly dependent on each other?"
        ),
        "criteria": [
            "Response correctly identifies Team A as experiencing a blocker on Tuesday.",
            "Response correctly identifies the blocker as missing API credentials.",
            "Response correctly identifies Friday as the day Team B began depending on Team A's API.",
            "Response does not attribute Team B's progress as blocked.",
            "Response addresses both team streams without omitting either.",
        ],
    },
    {
        "stream_a": (
            "Student A academic record: "
            "Assignment 1: Score 88/100. "
            "Midterm exam: Score 74/100. "
            "Assignment 2: Score 95/100. "
            "Assignment 3: Score 82/100. "
            "Final exam: Score 79/100."
        ),
        "stream_b": (
            "Student B academic record: "
            "Assignment 1: Score 91/100. "
            "Midterm exam: Score 85/100. "
            "Assignment 2: Score 78/100. "
            "Assignment 3: Score 94/100. "
            "Final exam: Score 88/100."
        ),
        "question": (
            "Tracking both student records simultaneously: "
            "Which student scored higher on the midterm exam, "
            "and which student has the higher average score across assignments only (excluding exams)?"
        ),
        "criteria": [
            "Response correctly identifies Student B as scoring higher on the midterm (85 vs 74).",
            "Response correctly calculates Student A's average assignment score as approximately 88.3 (265/3).",
            "Response correctly calculates Student B's average assignment score as approximately 87.7 (263/3).",
            "Response correctly identifies Student A as having the higher average assignment score.",
            "Response does not conflate exam scores with assignment scores or mix up the two students.",
        ],
    },
    {
        "stream_a": (
            "Factory A production log (Week 12): "
            "Monday: Produced 240 units, 3 defects reported. "
            "Tuesday: Produced 185 units, 0 defects reported. Line halted for 2 hours due to equipment calibration. "
            "Wednesday: Produced 310 units, 1 defect reported. "
            "Thursday: Produced 295 units, 5 defects reported. "
            "Friday: Produced 220 units, 2 defects reported."
        ),
        "stream_b": (
            "Factory B production log (Week 12): "
            "Monday: Produced 190 units, 1 defect reported. "
            "Tuesday: Produced 275 units, 4 defects reported. "
            "Wednesday: Produced 300 units, 0 defects reported. Line halted for 3 hours due to supply delay. "
            "Thursday: Produced 210 units, 2 defects reported. "
            "Friday: Produced 260 units, 1 defect reported."
        ),
        "question": (
            "Tracking both factory logs simultaneously: "
            "Which factory had higher total output for the week, "
            "which factory had more total defects, "
            "and what was the cause of each factory's production halt?"
        ),
        "criteria": [
            "Response correctly identifies Factory A as having higher total output (1250 units vs 1235 units).",
            "Response correctly identifies Factory A as having more total defects (11 vs 8).",
            "Response correctly attributes the equipment calibration halt to Factory A on Tuesday.",
            "Response correctly attributes the supply delay halt to Factory B on Wednesday.",
            "Response does not conflate the two factories' halt causes or production figures.",
        ],
    },
])


@kbench.task(name="divided_attention", store_task=False)
def divided_attention(
    llm,
    stream_a: str,
    stream_b: str,
    question: str,
    criteria: list,
):
    """
    Evaluating divided attention: model must simultaneously monitor and reason
    across two independent information streams without conflating them.
    Judge is a fixed external model.
    """
    prompt = (
        f"You are given two independent information streams. "
        f"Read both carefully and answer the question by tracking each stream separately.\n\n"
        f"Stream A:\n{stream_a}\n\n"
        f"Stream B:\n{stream_b}\n\n"
        f"Question: {question}"
    )

    response = llm.prompt(prompt)

    report = kbench.assertions.assess_response_with_judge(
        criteria=criteria,
        response_text=response,
        judge_llm=kbench.judge_llm,
    )
    for result in report.results:
        kbench.assertions.assert_true(
            result.passed,
            expectation=result.criterion,
        )


divided_attention.bind_dataframe(TASK_DATA, name="divided_attention", store_task=True)

Task(func=<function Task.bind_dataframe.<locals>.func at 0x7bec8f181da0>, name='divided_attention', description='Evaluating divided attention: model must simultaneously monitor and reason\nacross two independent information streams without conflating them.\nJudge is a fixed external model.', result_type=<class 'kaggle_benchmarks.results.PassCount'>, version=1, store_task=True, store_run=True)

In [3]:
all_runs = []

for model_name, model_llm in kbench.llms.items():
    print(f"\n--- {model_name} ---")
    for i, row in TASK_DATA.iterrows():
        run = divided_attention.run(
            llm=model_llm,
            stream_a=row["stream_a"],
            stream_b=row["stream_b"],
            question=row["question"],
            criteria=row["criteria"],
        )
        all_runs.append((model_name, i, run))
        passed = sum(ar.passed for ar in run.assertion_results)
        total = len(run.assertion_results)
        print(f"  Row {i}: {run.status.name} -- {passed}/{total} assertions passed")


--- anthropic/claude-haiku-4-5@20251001 ---
  Row 0: SUCCESS -- 5/5 assertions passed
  Row 1: SUCCESS -- 4/5 assertions passed
  Row 2: SUCCESS -- 5/5 assertions passed
  Row 3: SUCCESS -- 5/5 assertions passed
  Row 4: SUCCESS -- 5/5 assertions passed

--- anthropic/claude-opus-4-1@20250805 ---
  Row 0: SUCCESS -- 5/5 assertions passed
  Row 1: SUCCESS -- 5/5 assertions passed
  Row 2: SUCCESS -- 5/5 assertions passed
  Row 3: SUCCESS -- 5/5 assertions passed
  Row 4: SUCCESS -- 5/5 assertions passed

--- anthropic/claude-opus-4-5@20251101 ---
  Row 0: SUCCESS -- 5/5 assertions passed
  Row 1: SUCCESS -- 5/5 assertions passed
  Row 2: SUCCESS -- 5/5 assertions passed
  Row 3: SUCCESS -- 5/5 assertions passed
  Row 4: SUCCESS -- 5/5 assertions passed

--- anthropic/claude-opus-4-6@default ---
  Row 0: SUCCESS -- 5/5 assertions passed
  Row 1: SUCCESS -- 5/5 assertions passed
  Row 2: SUCCESS -- 5/5 assertions passed
  Row 3: SUCCESS -- 5/5 assertions passed
  Row 4: SUCCESS -- 5/5 as

In [4]:
def run_to_record(model_name, row_index, run):
    passed = sum(ar.passed for ar in run.assertion_results)
    total = len(run.assertion_results)
    assertion_summary = "; ".join(
        f"{'PASS' if ar.passed else 'FAIL'}: {ar.expectation}"
        for ar in run.assertion_results
    )
    return {
        "model": model_name,
        "row": row_index,
        "status": run.status.name,
        "assertions_passed": passed,
        "assertions_total": total,
        "pass_rate": round(passed / total, 2) if total > 0 else None,
        "error_message": run.error_message,
        "assertion_details": assertion_summary,
    }


records = [run_to_record(model_name, i, run) for model_name, i, run in all_runs]
results_df = pd.DataFrame(records)
pd.set_option("display.max_colwidth", 100)

print("\n=== Per-row results ===")
display(results_df)

print("\n=== Summary by model ===")
summary = (
    results_df.groupby("model")
    .agg(
        assertions_passed=("assertions_passed", "sum"),
        assertions_total=("assertions_total", "sum"),
    )
    .assign(pass_rate=lambda df: (df["assertions_passed"] / df["assertions_total"]).round(3))
    .sort_values("pass_rate", ascending=False)
    .reset_index()
)
display(summary)


=== Per-row results ===


,model,row,status,assertions_passed,assertions_total,pass_rate,error_message,assertion_details
0,anthropic/claude-haiku-4-5@20251001,0,SUCCESS,5,5,1.0,None,PASS: Response correctly identifies 2 errors on Server A.; PASS: Response correctly identifies 2...
1,anthropic/claude-haiku-4-5@20251001,1,SUCCESS,4,5,0.8,None,FAIL: Response correctly identifies GOOG and MSFT as appearing in both portfolios.; PASS: Respon...
2,anthropic/claude-haiku-4-5@20251001,2,SUCCESS,5,5,1.0,None,PASS: Response correctly identifies Team A as experiencing a blocker on Tuesday.; PASS: Response...
3,anthropic/claude-haiku-4-5@20251001,3,SUCCESS,5,5,1.0,None,PASS: Response correctly identifies Student B as scoring higher on the midterm (85 vs 74).; PASS...
4,anthropic/claude-haiku-4-5@20251001,4,SUCCESS,5,5,1.0,None,PASS: Response correctly identifies Factory A as having higher total output (1250 units vs 1235 ...
...,...,...,...,...,...,...,...,...
160,zai/glm-5,0,SUCCESS,5,5,1.0,None,PASS: Response correctly identifies 2 errors on Server A.; PASS: Response correctly identifies 2...
161,zai/glm-5,1,SUCCESS,5,5,1.0,None,PASS: Response correctly identifies GOOG and MSFT as appearing in both portfolios.; PASS: Respon...
162,zai/glm-5,2,SUCCESS,5,5,1.0,None,PASS: Response correctly identifies Team A as experiencing a blocker on Tuesday.; PASS: Response...
163,zai/glm-5,3,SUCCESS,5,5,1.0,None,PASS: Response correctly identifies Student B as scoring higher on the midterm (85 vs 74).; PASS...



=== Summary by model ===


,model,assertions_passed,assertions_total,pass_rate
0,anthropic/claude-opus-4-1@20250805,25,25,1.00
1,google/gemini-3.1-pro-preview,25,25,1.00
2,anthropic/claude-opus-4-5@20251101,25,25,1.00
3,anthropic/claude-opus-4-6@default,25,25,1.00
4,anthropic/claude-sonnet-4-5@20250929,25,25,1.00
5,anthropic/claude-sonnet-4-6@default,25,25,1.00
6,deepseek-ai/deepseek-v3.1,25,25,1.00
7,google/gemini-2.0-flash,25,25,1.00
8,deepseek-ai/deepseek-v3.2,25,25,1.00
9,qwen/qwen3-235b-a22b-instruct-2507,25,25,1.00


In [5]:
output_path = "divided_attention_results.csv"
summary.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
display(summary)

# Detailed per-row results (includes per-criterion pass/fail for deeper analysis)
detail_path = "divided_attention_detailed_results.csv"
results_df.to_csv(detail_path, index=False)
print(f"Saved detailed results: {detail_path}")

Saved: divided_attention_results.csv


,model,assertions_passed,assertions_total,pass_rate
0,anthropic/claude-opus-4-1@20250805,25,25,1.00
1,google/gemini-3.1-pro-preview,25,25,1.00
2,anthropic/claude-opus-4-5@20251101,25,25,1.00
3,anthropic/claude-opus-4-6@default,25,25,1.00
4,anthropic/claude-sonnet-4-5@20250929,25,25,1.00
5,anthropic/claude-sonnet-4-6@default,25,25,1.00
6,deepseek-ai/deepseek-v3.1,25,25,1.00
7,google/gemini-2.0-flash,25,25,1.00
8,deepseek-ai/deepseek-v3.2,25,25,1.00
9,qwen/qwen3-235b-a22b-instruct-2507,25,25,1.00


Saved detailed results: divided_attention_detailed_results.csv
